# 01 — Exploratory Data Analysis

This notebook checks data quality, target shape, demand patterns, relationships, and outliers before modelling.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd() if (Path.cwd() / "data/raw/train.csv").exists() else Path.cwd().parent
df = pd.read_csv(ROOT / "data/raw/train.csv", parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
df.head()

## Shape, data types, missing values and duplicates

In [ ]:
print("Shape:", df.shape)
display(df.dtypes.to_frame("dtype"))
display(df.isnull().sum().to_frame("missing values"))
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate timestamps:", df["datetime"].duplicated().sum())

## Target distribution and skew

Bike counts usually have a right tail because rush-hour and special-demand periods can be much busier than quiet hours. `log1p(count)` compresses extreme values and makes the regression target less skewed; predictions are converted back with `expm1`.

In [ ]:
print(df["count"].describe())
print("Target skew:", df["count"].skew())
print("log1p target skew:", np.log1p(df["count"]).skew())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["count"], bins=40, ax=ax[0]); ax[0].set_title("Hourly bike count")
sns.histplot(np.log1p(df["count"]), bins=40, ax=ax[1]); ax[1].set_title("log1p(hourly bike count)")
plt.tight_layout()

## Demand by hour, season, weather and working day

In [ ]:
df["hour"] = df["datetime"].dt.hour
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
sns.lineplot(data=df, x="hour", y="count", estimator="mean", errorbar=None, ax=ax[0,0]); ax[0,0].set_title("Average demand by hour")
sns.barplot(data=df, x="season", y="count", errorbar=None, ax=ax[0,1]); ax[0,1].set_title("Demand by season")
sns.barplot(data=df, x="weather", y="count", errorbar=None, ax=ax[1,0]); ax[1,0].set_title("Demand by weather")
sns.barplot(data=df, x="workingday", y="count", errorbar=None, ax=ax[1,1]); ax[1,1].set_title("Demand by working day")
plt.tight_layout()

## Correlation heatmap

`temp` and `atemp` measure closely related quantities and may be strongly correlated. This multicollinearity can affect coefficient interpretation for linear models, though it is less of a concern for tree models.

In [ ]:
numeric = df.select_dtypes(include="number")
plt.figure(figsize=(11, 8))
sns.heatmap(numeric.corr(), cmap="coolwarm", center=0)
plt.title("Numeric feature correlation heatmap")
plt.tight_layout()

## Outlier inspection

Boxplots help identify unusually high counts or weather values. High demand is not automatically an error; it may represent genuine commuting or event patterns.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(y=df["count"], ax=ax[0]); ax[0].set_title("Count outliers")
sns.boxplot(data=df[["temp", "atemp", "humidity", "windspeed"]], ax=ax[1]); ax[1].set_title("Weather feature outliers")
plt.tight_layout()

## EDA summary

- Data is sorted chronologically before splitting.
- `casual` and `registered` describe components of the target count, so they are excluded from predictors to prevent leakage.
- `day` and `day_of_year` are not one-hot encoded because they can encourage date memorisation.
- Log-transforming the regression target can reduce the effect of a right-skewed target distribution.